# 🚧 Pipeline de Dados - Landing → Bronze
### Ingestão e armazenamento de dados no Data Lake

## 📥 Leitura dos Dados (Camada Landing)


In [0]:

# Mostra todas as tabelas do schema
spark.sql("SHOW TABLES IN acidentes.default").show()

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS acidentes.bronze")

In [0]:
df = spark.table("acidentes.default.datatran_2025")
display(df)

In [0]:
df.printSchema()

## 💾 Armazenamento no Data Lake com Governança e Rastreabilidade dos Dados (Bronze)

In [0]:
from pyspark.sql import functions as F
import uuid

def ingest_from_table(nome_tabela_origem: str, nome_tabela_destino: str):
    try:
        origem = f"acidentes.default.{nome_tabela_origem}"
        destino = f"acidentes.bronze.{nome_tabela_destino}"

        print(f"📥 Origem: {origem}")
        print(f"📍 Destino: {destino}")

        # 📖 Leitura da tabela (landing)
        df = spark.table(origem)

        if df.limit(1).count() == 0:
            raise ValueError("❌ Tabela vazia")

        # 🏷️ Padronização leve (Bronze)
        df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

        # 🔥 Versionamento
        ingestion_id = str(uuid.uuid4())

        df = (
            df
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("ingestion_date", F.current_date())  # ✅ FALTAVA ISSO
            .withColumn("ingestion_id", F.lit(ingestion_id))
            .withColumn("source_table", F.lit(nome_tabela_origem))
        )

        # 💾 Escrita Bronze
        (
            df.write
            .format("delta")
            .mode("append")
            .partitionBy("ingestion_date")  # agora existe
            .option("mergeSchema", "true")
            .saveAsTable(destino)
        )

        # 🧾 LOG DE INGESTÃO (COLOCA AQUI 👇)
        qtd_linhas = df.count()

        log_df = spark.createDataFrame([(
            ingestion_id,
            nome_tabela_origem,
            nome_tabela_destino,
            qtd_linhas
        )], ["ingestion_id", "source_table", "target_table", "row_count"])

        log_df = log_df.withColumn("log_timestamp", F.current_timestamp())

        log_df.write.mode("append").saveAsTable("acidentes.bronze.ingestion_log")

        print(f"✅ Ingestion ID: {ingestion_id}")
        print("✅ Bronze criada com sucesso!\n")

    except Exception as e:
        print(f"⚠️ Erro: {str(e)}")

In [0]:
ingest_from_table("datatran_2025", "acidentes_2025")

## ✅ Validação da Camada Bronze

In [0]:
df = spark.table("acidentes.bronze.acidentes_2025")
display(df)

In [0]:
df.printSchema()

In [0]:
log_df = spark.table("acidentes.bronze.ingestion_log")
display(log_df)